In [1]:
from models import model, summarization_model
from langchain.agents import create_agent, AgentState
from langchain.messages import RemoveMessage, ToolMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langgraph.checkpoint.memory import InMemorySaver

In [2]:
@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, any] | None:
    """Remove all the tool messages from the state"""
    messages = state['messages']
    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    return {'messages': [RemoveMessage(id=m.id) for m in tool_messages]}

In [3]:
agent = create_agent(
    model,
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [4]:
response = agent.invoke(
    {
        "messages": [
            ('human', "My device won't turn on. What should I do?"),
            ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
            ('ai', "Is the device plugged in and turned on?"),
            ('human', "Yes, it's plugged in and turned on."),
            ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
            ('ai', "Is the device showing any lights or indicators?"),
            ('human', "Yes, it has a red light on")
            # ('human', "What's the temperature of the device?")
        ]
    },
    {"configurable": {"thread_id": "xyz"}}
)

response

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='3954626b-df8a-4eec-a526-b80746b95dcb'),
  AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='fe77d709-e20a-4d92-93fc-35f2d2647761', tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='957fc650-ef59-4fd1-be21-7c446c3013c2'),
  AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='9f6f55fb-bd9c-4c08-abea-29acdd685fb5', tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='Yes, it has a red light on', additional_kwargs={}, response_metadata={}, id='baa9be3e-19dc-46fd-9d13-a804217a1c0f'),
  AIMessage(content='A red light can mean a few different things depending on the type of device you’re using (laptop, desktop, tablet, router, etc.). Bel

In [5]:
response['messages'][-1].content

'A red light can mean a few different things depending on the type of device you’re using (laptop, desktop, tablet, router, etc.). Below is a step‑by‑step troubleshooting checklist that works for most electronics. Follow the steps in order, and let me know what you see after each one so I can narrow down the issue.\n\n---\n\n## 1️⃣ Identify the Device & Context\n| Question | Why it matters |\n|----------|----------------|\n| **What exact model** (brand + model number) is it? | Red‑light codes are often model‑specific. |\n| **What does the device normally do** (e.g., computer, router, game console, monitor, printer)? | Different devices have different “red‑light” meanings. |\n| **When did the problem start?** (after a power outage, update, drop, spill, etc.) | Helps pinpoint whether it’s a software glitch or hardware failure. |\n| **Any recent changes** (new software, accessories, firmware update, moved the device, etc.)? | Recent changes can trigger a fault. |\n\n*If you can share the 